# CSV Data Cleaning - Review SY-08002944

**Objective**: Clean and organize data from Review_SY-08002944_4_3_2025 10_31_21.csv

**Data Source**: Code/DSCwashumed/backend/data/Review_SY-08002944_4_3_2025 10_31_21.csv

**Output**: Cleaned data files in cleadned_data folder

**Process**: Load → Analyze → Clean → Export

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Data paths
data_file = Path("Review_SY-08002944_4_3_2025 10_31_21.csv")
output_dir = Path("cleadned_data")
output_dir.mkdir(exist_ok=True)

print(f"Data file: {data_file}")
print(f"Output directory: {output_dir}")
print(f"Data file exists: {data_file.exists()}")

Data file: Review_SY-08002944_4_3_2025 10_31_21.csv
Output directory: cleadned_data
Data file exists: True


## Data Loading and Initial Analysis

In [2]:
# Load the CSV data
def load_csv_data():
    try:
        df = pd.read_csv(data_file)
        print(f"Shape: {df.shape}")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# Load data
raw_df = load_csv_data()

if raw_df is not None:
    raw_df.head()

Shape: (25, 43)


## Data Quality Assessment

In [3]:
# Analyze data quality
def analyze_data_quality(df):
    missing_counts = df.isnull().sum()
    missing_percent = (missing_counts / len(df)) * 100
    duplicates = df.duplicated().sum()
    
    print(f"Missing values by column:")
    for col in df.columns:
        if missing_counts[col] > 0:
            print(f"  {col}: {missing_counts[col]} ({missing_percent[col]:.1f}%)")
    
    print(f"Duplicate rows: {duplicates}")
    
    return missing_counts, missing_percent

if raw_df is not None:
    missing_counts, missing_percent = analyze_data_quality(raw_df)

Missing values by column:
  Patient ID: 25 (100.0%)
  Patient: 25 (100.0%)
  Owner Last Name: 25 (100.0%)
  Gender: 25 (100.0%)
  Age: 25 (100.0%)
  Draw Date: 25 (100.0%)
  Draw Time: 25 (100.0%)
  Delivery Date: 25 (100.0%)
  Delivery Time: 25 (100.0%)
  Veterinarian: 25 (100.0%)
  Comments: 25 (100.0%)
  WBC Message: 10 (40.0%)
  RBC Message: 20 (80.0%)
  PLT Message: 21 (84.0%)
  Unnamed: 42: 25 (100.0%)
Duplicate rows: 0


## Data Cleaning

In [4]:
# Clean the data
def clean_csv_data(df):
    df_clean = df.copy()
    
    # Remove duplicate rows
    df_clean = df_clean.drop_duplicates()
    
    # Clean column names
    df_clean.columns = df_clean.columns.str.strip().str.replace(' ', '_').str.replace('[^a-zA-Z0-9_]', '', regex=True)
    
    # Handle missing values for lab data appropriately
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    
    # For numeric lab values, use median imputation
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            if df_clean[col].notna().sum() > 0:
                median_val = df_clean[col].median()
                df_clean[col] = df_clean[col].fillna(median_val)
            else:
                df_clean[col] = df_clean[col].fillna(0)
    
    # For categorical values, use "Unknown"
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col] = df_clean[col].fillna("Unknown")
    
    # Convert data types where appropriate
    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            try:
                numeric_series = pd.to_numeric(df_clean[col], errors='coerce')
                if not numeric_series.isnull().all():
                    df_clean[col] = numeric_series
            except:
                pass
    
    return df_clean

if raw_df is not None:
    cleaned_df = clean_csv_data(raw_df)
    cleaned_df.head()

## Data Validation

In [5]:
# Validate cleaned data
def validate_cleaned_data(df):
    missing_values = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()
    
    print(f"Missing values: {missing_values}")
    print(f"Duplicate rows: {duplicate_rows}")
    print(f"Shape: {df.shape}")
    
    return df.describe()

if 'cleaned_df' in locals() and cleaned_df is not None:
    summary_stats = validate_cleaned_data(cleaned_df)
    summary_stats

Missing values: 119
Duplicate rows: 0
Shape: (25, 43)


## Export Cleaned Data

In [6]:
# Export cleaned data to cleadned_data folder
def export_cleaned_data(df):
    base_name = "Review_SY-08002944_cleaned"
    csv_output = output_dir / f"{base_name}.csv"
    excel_output = output_dir / f"{base_name}.xlsx"
    
    # Export to CSV
    df.to_csv(csv_output, index=False)
    
    # Export to Excel with multiple sheets
    with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Cleaned_Data', index=False)
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            summary_stats = df[numeric_cols].describe()
            summary_stats.to_excel(writer, sheet_name='Summary_Statistics')
        
        info_data = {
            'Column': df.columns,
            'Data_Type': [str(dtype) for dtype in df.dtypes],
            'Non_Null_Count': [df[col].count() for col in df.columns],
            'Null_Count': [df[col].isnull().sum() for col in df.columns],
            'Unique_Values': [df[col].nunique() for col in df.columns]
        }
        info_df = pd.DataFrame(info_data)
        info_df.to_excel(writer, sheet_name='Data_Info', index=False)
    
    print(f"Files exported to {output_dir}")
    return csv_output, excel_output

if 'cleaned_df' in locals() and cleaned_df is not None:
    csv_path, excel_path = export_cleaned_data(cleaned_df)

Files exported to cleadned_data


In [7]:
# Display cleaned data structure
print("=== CLEANED DATA OVERVIEW ===")
print(f"Shape: {cleaned_df.shape}")
print(f"Columns: {list(cleaned_df.columns)}")
print("\n=== SAMPLE DATA ===")
print(cleaned_df.head())

print("\n=== COLUMN DATA TYPES ===")
print(cleaned_df.dtypes)

print("\n=== NUMERIC COLUMNS SUMMARY ===")
numeric_cols = cleaned_df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(cleaned_df[numeric_cols].describe())

=== CLEANED DATA OVERVIEW ===
Shape: (25, 43)
Columns: ['Sample_ID', 'Patient_ID', 'Patient', 'WBC_103uL', 'Neu__103uL', 'Lym__103uL', 'Mon__103uL', 'Eos__103uL', 'Bas__103uL', 'Neu__', 'Lym__', 'Mon__', 'Eos__', 'Bas__', 'RBC_106uL', 'HGB_gdL', 'HCT_', 'MCV_fL', 'MCH_pg', 'MCHC_gdL', 'RDWCV_', 'PLT_103uL', 'MPV_fL', 'Species', 'Sample_State', 'Owner_Last_Name', 'Mode', 'Date', 'Time', 'Gender', 'Age', 'Ref_Group', 'Draw_Date', 'Draw_Time', 'Delivery_Date', 'Delivery_Time', 'Veterinarian', 'Operator', 'Comments', 'WBC_Message', 'RBC_Message', 'PLT_Message', 'Unnamed_42']

=== SAMPLE DATA ===
   Sample_ID  Patient_ID  Patient  WBC_103uL  Neu__103uL  Lym__103uL  \
0     5410.0         0.0      0.0        NaN        2.50         NaN   
1     5409.0         0.0      0.0        NaN        1.41         NaN   
2     5408.0         0.0      0.0        NaN        2.46         NaN   
3     5407.0         0.0      0.0        NaN        3.31         NaN   
4     5406.0         0.0      0.0        